# Stage 5: Production retrieval pipeline

This notebook inspects the frozen retrieval configuration, demonstrates non-oracle query analysis, and summarizes development and sealed-test results. Production logic remains in `src/`.

In [ ]:
import json
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
PROCESSED = PROJECT_ROOT / 'data' / 'processed'
assert (PROCESSED / 'retrieval_config.json').exists()

## Exact, ambiguity-preserving query analysis

These examples are ordinary demonstration text, not evaluation queries.

In [ ]:
from src.alias_resolver import AliasResolver
from src.dataset_io import read_json, read_jsonl
from src.query_analyzer import QueryAnalyzer

corpus = read_jsonl(PROCESSED / 'knowledge_corpus.jsonl')
analyzer = QueryAnalyzer(AliasResolver(corpus))
examples = [
    'Plan Mathematics revision for today',
    'How should I use flashcards for vocabulary?',
    'The posture sensor reports uneven pressure while sitting',
    'Tell me about IT',
]
analyses = [analyzer.analyze(text) for text in examples]
assert analyses[0].alias_resolution.resolved_subject_record_ids
assert analyses[2].primary_intent == 'sensor_or_posture'
assert analyses[3].alias_resolution.has_ambiguity
[(text, item.primary_intent, item.alias_resolution.has_ambiguity) for text, item in zip(examples, analyses)]

## Frozen development decision

The 64-query development comparison selected exact alias resolution plus dense retrieval before final-test evaluation.

In [ ]:
frozen = read_json(PROCESSED / 'retrieval_config.json')
assert frozen['frozen'] is True
assert frozen['selected_config_id'] == 'alias_plus_dense'
development_summary = {
    name: {
        'recall_at_5': values['overall']['recall_at_5'],
        'mrr_at_10': values['overall']['mrr_at_10'],
    }
    for name, values in frozen['development_candidate_metrics'].items()
}
development_summary

## Sealed final-test result

The final artifact records the frozen configuration hash and the split-isolation checks from the single guarded run.

In [ ]:
final_metrics = read_json(PROCESSED / 'final_retrieval_metrics.json')
assert final_metrics['query_count'] == 32
assert final_metrics['development_isolation']['final_test_ids_in_development_artifacts'] == 0
assert final_metrics['retrieval_configuration_hash'] == frozen['retrieval_configuration_hash']
{
    key: final_metrics['metrics']['overall'][key]
    for key in ('recall_at_1', 'recall_at_3', 'recall_at_5', 'recall_at_10', 'mrr_at_10', 'ndcg_at_10')
}